In [3]:
import os, sys
from pathlib import Path
from dotenv import load_dotenv

# Locate repo root and add project/ to path
REPO_ROOT = next(p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / '.git').exists())
PROJECT_DIR = REPO_ROOT / 'project'
sys.path.insert(0, str(PROJECT_DIR))

load_dotenv(REPO_ROOT / '.env')

TOOLBENCH_DIR = Path(os.environ.get('TOOLBENCH_DIR', str(REPO_ROOT / 'toolbench_data')))
print(f"Repo:      {REPO_ROOT}")
print(f"Toolbench: {TOOLBENCH_DIR}")

Repo:      /Users/michaelserrano/Documents/GitHub/mmai
Toolbench: /Users/michaelserrano/Documents/GitHub/mmai/toolbench_data


In [4]:
from data.load_toolbench import load_api_corpus
from models.embeddings import format_api_string, get_embeddings
import numpy as np
from tqdm import tqdm

corpus = load_api_corpus(TOOLBENCH_DIR / "toolenv" / "tools")
print(f"Loaded {len(corpus)} APIs across {len(set(a['category'] for a in corpus))} categories")

Loaded 49937 APIs across 50 categories


In [5]:
api_strings = [format_api_string(a) for a in corpus]
print("Sample:", api_strings[0])

# Embed all APIs (cached — safe to re-run)
BATCH = 500
all_embeddings = []
for i in tqdm(range(0, len(api_strings), BATCH)):
    batch = api_strings[i:i+BATCH]
    embs = get_embeddings(batch)
    all_embeddings.append(embs)

corpus_matrix = np.vstack(all_embeddings)
np.save(PROJECT_DIR / "corpus_embeddings.npy", corpus_matrix)
print(f"✓ Saved corpus_embeddings.npy: shape {corpus_matrix.shape}")

Sample: Advertising > a: api — a | Parameters:


100%|██████████| 100/100 [04:19<00:00,  2.60s/it]


✓ Saved corpus_embeddings.npy: shape (49937, 1536)


In [6]:
import json
api_names = [a["action_name"] for a in corpus]
with open(PROJECT_DIR / "api_names.json", "w") as f:
    json.dump(api_names, f)
print(f"✓ Saved {len(api_names)} API names")

✓ Saved 49937 API names
